# Fase 4 — Entrenamiento y evaluación del detector YOLO

Este notebook implementa el flujo reproducible de entrenamiento y evaluación del detector YOLO para OptiSignal-AI.

**Objetivo:** Entrenar modelos YOLO independientes para LISA y Bosch, evaluar Precision, Recall y mAP@50 y guardar los pesos.

**Hardware:** GPU NVIDIA con CUDA obligatoria para entrenamiento local. El notebook no continuará silenciosamente por CPU si CUDA falla. En Colab, selecciona primero un runtime con GPU.

**Antes de ejecutar:** En VS Code selecciona el kernel correcto. Instala PyTorch con CUDA desde la terminal de ese mismo Python (instrucciones en `PHASE_4_YOLO_TRAINING.md`), reinicia el kernel y después ejecuta las celdas en orden. La celda de dependencias del notebook no reinstala PyTorch.


## 1. Instalación de dependencias

In [1]:
import sys
import os

IS_COLAB = 'google.colab' in sys.modules
print(f'Entorno: {"Google Colab" if IS_COLAB else "Local"}')

Entorno: Local


In [2]:
import subprocess

# PyTorch debe instalarse previamente con su build CUDA desde el Python del kernel.
# Así evitamos que pip instale/reemplace accidentalmente una build solo CPU.
packages = [
    'ultralytics>=8.0.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pandas>=2.0.0',
    'matplotlib>=3.7.0',
    'Pillow>=10.0.0',
    'psutil>=5.9.0'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencias de visión instaladas. PyTorch no se modificó en esta celda.')
print(f'Python del kernel: {sys.executable}')


Dependencias de visión instaladas. PyTorch no se modificó en esta celda.
Python del kernel: c:\Users\nicpac\AppData\Local\Programs\Python\Python312\python.exe


## 2. Detección de hardware y configuración de dispositivo

In [3]:
import torch
import platform
import psutil

print('=== INFORMACIÓN DEL SISTEMA ===')
print(f'Sistema operativo: {platform.system()} {platform.release()}')
print(f'Procesador: {platform.processor()}')
print(f'RAM disponible: {psutil.virtual_memory().available / (1024**3):.2f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'Build CUDA de PyTorch: {torch.version.cuda}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó CUDA. El entrenamiento se detuvo para evitar usar CPU. '
        'Instala una build CUDA de PyTorch en el Python de este kernel, reinicia el kernel '
        'y vuelve a ejecutar. Consulta PHASE_4_YOLO_TRAINING.md.'
    )

try:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.empty(1, device='cuda:0')
    torch.cuda.synchronize()
except Exception as exc:
    raise RuntimeError(f'PyTorch ve CUDA, pero no pudo inicializar la GPU: {exc}') from exc

device = '0'
print(f'GPU detectada: {gpu_name}')
print(f'Memoria GPU: {gpu_memory:.2f} GB')
print('Dispositivo seleccionado: CUDA:0')


=== INFORMACIÓN DEL SISTEMA ===
Sistema operativo: Windows 11
Procesador: AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
RAM disponible: 3.50 GB
PyTorch: 2.14.0+cu130
Build CUDA de PyTorch: 13.0
CUDA disponible: True
GPU detectada: NVIDIA GeForce RTX 4060 Ti
Memoria GPU: 8.00 GB
Dispositivo seleccionado: CUDA:0


## 3. Configuración de parámetros

In [4]:
from pathlib import Path

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/optisignal-ai')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_ROOT = PROJECT_ROOT / 'data' / 'processed'
MODELS_ROOT = PROJECT_ROOT / 'models'
RESULTS_ROOT = PROJECT_ROOT / 'results' / 'phase_4'

MODELS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

YOLO_MODEL = 'yolov8n'
EPOCHS = 5
BATCH_SIZE = 8
IMG_SIZE = 640
PATIENCE = 10
SEED = 42
DATASETS = ['lisa', 'bosch']

print('=== CONFIGURACIÓN ===')
print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'Datos procesados: {DATA_ROOT}')
print(f'Modelos: {MODELS_ROOT}')
print(f'Resultados: {RESULTS_ROOT}')
print(f'\nModelo YOLO: {YOLO_MODEL}')
print(f'Épocas: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Tamaño imagen: {IMG_SIZE}x{IMG_SIZE}')
print(f'Datasets: {DATASETS}')

=== CONFIGURACIÓN ===
Raíz del proyecto: c:\Users\nicpac\optisignal-ai
Datos procesados: c:\Users\nicpac\optisignal-ai\data\processed
Modelos: c:\Users\nicpac\optisignal-ai\models
Resultados: c:\Users\nicpac\optisignal-ai\results\phase_4

Modelo YOLO: yolov8n
Épocas: 5
Batch size: 8
Tamaño imagen: 640x640
Datasets: ['lisa', 'bosch']


## 4. Validación de estructura de datos

In [5]:
import json

print('=== VALIDACIÓN DE DATASETS ===')

for dataset_name in DATASETS:
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    
    if not dataset_path.exists():
        print(f'\n✗ {dataset_name.upper()}: No encontrado en {dataset_path}')
        continue
    
    required_dirs = ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val', 'labels/test']
    missing = [d for d in required_dirs if not (dataset_path / d).exists()]
    
    if missing:
        print(f'\n✗ {dataset_name.upper()}: Directorios faltantes: {missing}')
        continue
    
    train_count = len(list((dataset_path / 'images/train').glob('*.[jp][pn]g')))
    val_count = len(list((dataset_path / 'images/val').glob('*.[jp][pn]g')))
    test_count = len(list((dataset_path / 'images/test').glob('*.[jp][pn]g')))
    total = train_count + val_count + test_count
    
    manifest_path = dataset_path / 'partition_manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            manifest = json.load(f)
        print(f'\n✓ {dataset_name.upper()}')
        print(f'  Total imágenes: {total}')
        print(f'  Train: {train_count} ({train_count/total*100:.1f}%)')
        print(f'  Val:   {val_count} ({val_count/total*100:.1f}%)')
        print(f'  Test:  {test_count} ({test_count/total*100:.1f}%)')
        print(f'  Seed: {manifest.get("seed", "N/A")}')
    else:
        print(f'\n⚠ {dataset_name.upper()}: Manifiesto no encontrado')
        print(f'  Total imágenes: {total}')
        print(f'  Train: {train_count}, Val: {val_count}, Test: {test_count}')

=== VALIDACIÓN DE DATASETS ===

✓ LISA
  Total imágenes: 36775
  Train: 25742 (70.0%)
  Val:   7355 (20.0%)
  Test:  3678 (10.0%)
  Seed: 42

✓ BOSCH
  Total imágenes: 7147
  Train: 5002 (70.0%)
  Val:   1429 (20.0%)
  Test:  716 (10.0%)
  Seed: 42


## 5. Entrenamiento del modelo YOLO

In [6]:
from ultralytics import YOLO

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

training_results = {}

for dataset_name in DATASETS:
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    data_yaml = dataset_path / 'data.yaml'
    
    if not data_yaml.exists():
        print(f'\n✗ {dataset_name.upper()}: data.yaml no encontrado')
        continue
    
    print(f'\n=== ENTRENAMIENTO {dataset_name.upper()} ===')
    print(f'Configuración:')
    print(f'  Modelo: {YOLO_MODEL}')
    print(f'  Épocas: {EPOCHS}')
    print(f'  Batch size: {BATCH_SIZE}')
    print(f'  Dispositivo: {device.upper()}')
    print(f'  Data YAML: {data_yaml}')
    
    model = YOLO(f'{YOLO_MODEL}.pt')
    
    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=device,
        patience=PATIENCE,
        seed=SEED,
        project=str(RESULTS_ROOT / dataset_name),
        name='train',
        exist_ok=True,
        verbose=True,
        save=True,
        save_period=5
    )
    
    best_weights = RESULTS_ROOT / dataset_name / 'train' / 'weights' / 'best.pt'
    if best_weights.exists():
        model_output = MODELS_ROOT / f'{dataset_name}_best.pt'
        import shutil
        shutil.copy2(best_weights, model_output)
        print(f'\n✓ Pesos guardados en {model_output}')
    
    training_results[dataset_name] = {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'device': device,
        'results_path': str(RESULTS_ROOT / dataset_name / 'train')
    }
    
    print(f'\n✓ Entrenamiento {dataset_name.upper()} completado')

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\nicpac\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

=== ENTRENAMIENTO LISA ===
Configuración:
  Modelo: yolov8n
  Épocas: 5
  Batch size: 8
  Dispositivo: 0
  Data YAML: c:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\data.yaml
Ultralytics 8.4.164  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8187MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\data.y

## 6. Evaluación del modelo

In [7]:
evaluation_results = {}

for dataset_name in DATASETS:
    model_path = MODELS_ROOT / f'{dataset_name}_best.pt'
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    data_yaml = dataset_path / 'data.yaml'
    
    if not model_path.exists():
        print(f'\n⚠ {dataset_name.upper()}: Modelo no encontrado en {model_path}')
        continue
    
    print(f'\n=== EVALUACIÓN {dataset_name.upper()} ===')
    
    model = YOLO(str(model_path))
    metrics = model.val(data=str(data_yaml), device=device)
    
    eval_data = {
        'dataset': dataset_name,
        'model': YOLO_MODEL,
        'device': device,
        'metrics': {
            'precision': float(metrics.box.mp) if hasattr(metrics, 'box') else None,
            'recall': float(metrics.box.mr) if hasattr(metrics, 'box') else None,
            'mAP50': float(metrics.box.map50) if hasattr(metrics, 'box') else None,
            'mAP50_95': float(metrics.box.map) if hasattr(metrics, 'box') else None
        }
    }
    
    evaluation_results[dataset_name] = eval_data
    
    print(f'Precision: {eval_data["metrics"]["precision"]:.4f}' if eval_data['metrics']['precision'] else 'Precision: N/A')
    print(f'Recall: {eval_data["metrics"]["recall"]:.4f}' if eval_data['metrics']['recall'] else 'Recall: N/A')
    print(f'mAP@50: {eval_data["metrics"]["mAP50"]:.4f}' if eval_data['metrics']['mAP50'] else 'mAP@50: N/A')
    print(f'mAP@50-95: {eval_data["metrics"]["mAP50_95"]:.4f}' if eval_data['metrics']['mAP50_95'] else 'mAP@50-95: N/A')
    
    results_file = RESULTS_ROOT / f'{dataset_name}_evaluation.json'
    with open(results_file, 'w') as f:
        json.dump(eval_data, f, indent=2)
    print(f'\n✓ Resultados guardados en {results_file}')


=== EVALUACIÓN LISA ===
Ultralytics 8.4.164  Python-3.12.10 torch-2.14.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8187MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 187.3108.9 MB/s, size: 157.9 KB)
val: Scanning C:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\labels\val.cache... 7355 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 7355/7355  0.0s
val: C:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\images\val\dayClip6--00002_c21992dc41.jpg: 4 duplicate labels removed
val: C:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\images\val\dayClip6--00010_e94bc15f13.jpg: 5 duplicate labels removed
val: C:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\images\val\dayClip6--00011_a3e7c51f5b.jpg: 4 duplicate labels removed
val: C:\Users\nicpac\optisignal-ai\data\processed\lisa\partitioned\images\val\dayClip6--00017_7e85faa17c.jpg: 6 duplicate labels remov

## 7. Resumen y próximos pasos

In [8]:
print('\n' + '='*70)
print('RESUMEN DE FASE 4 - ENTRENAMIENTO YOLO')
print('='*70)

print(f'\nConfiguraciones utilizadas:')
print(f'  Modelo: {YOLO_MODEL}')
print(f'  Épocas: {EPOCHS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Dispositivo: {device.upper()}')
print(f'  Seed: {SEED}')

print(f'Archivos generados:')
for dataset_name in DATASETS:
    model_path = MODELS_ROOT / f'{dataset_name}_best.pt'
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024**2)
        print(f'  ✓ {model_path.name} ({size_mb:.1f} MB)')
    
    eval_file = RESULTS_ROOT / f'{dataset_name}_evaluation.json'
    if eval_file.exists():
        print(f'  ✓ {eval_file.name}')

print(f'\nDirectorio de resultados: {RESULTS_ROOT}')

print(f'\nPróximos pasos:')
print(f'  1. Revisar métricas en {RESULTS_ROOT}')
print(f'  2. Ajustar hiperparámetros si es necesario')
print(f'  3. Proceder a Fase 5: Clasificación de colores (HSV)')
print(f'  4. Integrar con máquina de estados (Fase 6)')

print('='*70)


RESUMEN DE FASE 4 - ENTRENAMIENTO YOLO

Configuraciones utilizadas:
  Modelo: yolov8n
  Épocas: 5
  Batch size: 8
  Dispositivo: 0
  Seed: 42
Archivos generados:
  ✓ lisa_best.pt (5.9 MB)
  ✓ lisa_evaluation.json
  ✓ bosch_best.pt (5.9 MB)
  ✓ bosch_evaluation.json

Directorio de resultados: c:\Users\nicpac\optisignal-ai\results\phase_4

Próximos pasos:
  1. Revisar métricas en c:\Users\nicpac\optisignal-ai\results\phase_4
  2. Ajustar hiperparámetros si es necesario
  3. Proceder a Fase 5: Clasificación de colores (HSV)
  4. Integrar con máquina de estados (Fase 6)
